In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from lifelines import CoxPHFitter
from sklearn.preprocessing import RobustScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
response_OUS.isna().sum().sum()

0

In [4]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [5]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [6]:
# Check null values in D3
clinical_train.isnull().sum().sum()

0

In [7]:
clinical_train

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,event_OS
0,54.238356,1,0,1,0,0,1,0.0,0,0.000000,...,0.029974,0.815962,0.000000,0.002035,0.140311,0.000062,0.009991,0.000370,114.312329,0
1,54.539726,0,0,0,0,1,0,0.0,1,27.404795,...,0.059728,0.707300,0.000000,0.008732,0.191058,0.000349,0.023402,0.000699,111.846575,0
2,59.019178,0,1,0,0,0,1,0.0,1,41.019178,...,0.039463,0.819828,0.000034,0.002518,0.126531,0.000000,0.009452,0.000414,49.479452,1
3,70.726027,0,0,0,0,1,0,0.0,1,37.500000,...,0.061133,0.716212,0.000000,0.003296,0.192388,0.000000,0.018879,0.000899,17.589041,1
4,67.865753,0,0,0,0,1,0,0.0,1,53.000000,...,0.058589,0.696493,0.000199,0.008968,0.202073,0.000399,0.029892,0.001395,66.739726,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,60.435616,0,0,1,0,0,1,1.0,0,0.000000,...,0.029444,0.804831,0.000000,0.001442,0.152626,0.000000,0.009734,0.000361,42.805479,0
135,68.794521,0,0,1,0,0,1,1.0,0,0.000000,...,0.046451,0.792151,0.000000,0.004094,0.142778,0.000079,0.011810,0.000630,42.739726,0
136,57.498630,0,0,1,0,0,1,1.0,1,39.498630,...,0.016316,0.832202,0.000000,0.000914,0.140582,0.000000,0.009137,0.000522,42.312329,0
137,65.684932,0,0,1,0,0,1,1.0,1,71.527397,...,0.038862,0.787588,0.000000,0.003144,0.156640,0.000054,0.011978,0.000705,42.246575,0


## Test dataset: MAASTRO 

In [8]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [9]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [10]:
rows_with_nan = response_MAASTRO[response_MAASTRO.isna().any(axis=1)]

print("Rows with NaN values:")
print(rows_with_nan)

Rows with NaN values:
    patient_id  OS  OS_event  LRC  LRC_event  DFS  DFS_event
10          11 NaN       NaN  NaN        NaN  NaN        NaN
20          21 NaN       NaN  NaN        NaN  NaN        NaN
31          32 NaN       NaN  NaN        NaN  NaN        NaN
51          52 NaN       NaN  NaN        NaN  NaN        NaN
83          84 NaN       NaN  NaN        NaN  NaN        NaN
86          87 NaN       NaN  NaN        NaN  NaN        NaN


In [11]:
# Assuming your data is stored in a list or a pandas DataFrame/Series
# Convert data to a set for faster membership checking
data_set = set(list(response_MAASTRO['patient_id']))

# Find numbers from 1 to 114 not present in data
missing_numbers = set(range(1, 115)) - data_set

# Convert missing_numbers back to a sorted list
missing_numbers_list = sorted(missing_numbers)

In [12]:
MAASTRO_D3

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_003_PET,LBP_012_PET,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET
0,1,55,0,0,1,0,0,1,1,1,...,0.000026,0.001181,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341
1,2,55,0,0,1,0,0,0,0,0,...,0.000056,0.002735,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335
2,3,55,0,0,1,0,0,0,0,1,...,0.000286,0.001888,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229
3,4,61,1,0,0,0,1,1,0,1,...,0.000160,0.003037,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240
4,6,70,0,0,1,0,0,1,1,1,...,0.000071,0.000881,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.000000,0.000544,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078
95,111,63,0,0,0,0,1,0,0,1,...,0.000109,0.000869,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489
96,112,63,0,0,1,0,0,1,1,1,...,0.000000,0.000734,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141
97,113,54,0,0,1,0,0,1,1,0,...,0.000114,0.001640,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038


In [13]:
# Assuming your data is stored in a list or a pandas DataFrame/Series
# Convert data to a set for faster membership checking
data_set = set(list(MAASTRO_D3['patient_id']))

# Find numbers from 1 to 114 not present in data
missing_numbers = set(range(1, 115)) - data_set

# Convert missing_numbers back to a sorted list
missing_numbers_list = sorted(missing_numbers)

print("Numbers from 1 to 114 not present in data:", missing_numbers_list)


Numbers from 1 to 114 not present in data: [5, 9, 11, 21, 32, 36, 46, 52, 65, 76, 78, 81, 84, 87, 92]


In [14]:
# Assuming your data is stored in a list or a pandas DataFrame/Series
# Convert data to a set for faster membership checking
data_set = set(list(response_MAASTRO['patient_id']))

# Find numbers from 1 to 114 not present in data
missing_numbers = set(range(1, 115)) - data_set

# Convert missing_numbers back to a sorted list
missing_numbers_list = sorted(missing_numbers)

print("Numbers from 1 to 114 not present in data:", missing_numbers_list)


Numbers from 1 to 114 not present in data: []


In [15]:
# need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [16]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]

In [17]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [18]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [19]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [20]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]
lower, upper = np.percentile(y_MAASTRO['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 388)
y_train:  (139,)


(99, 390)

# Feature Selection: RENT

In [21]:
selected_features = ["hpv_related",
                    "uicc8_III-IV",
                    "shape_Sphericity",
                    "oropharynx",
                    "LBP_102_PET",
                    "pack_years"]


# Selecting features in the DataFrame
X_rent = X[selected_features]
X_new = X_rent.copy()

X_MAASTRO_rent = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_rent.copy()

# Standardization

In [22]:
original_X = X.copy()

In [23]:
# Standardize X_new, the new data with the selected features only 
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = RobustScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [24]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [25]:
X_new

,hpv_related,uicc8_III-IV,shape_Sphericity,oropharynx,LBP_102_PET,pack_years
0,0.0,0.0,0.761164,1,0.000000,0.000000
1,0.0,0.0,0.697049,0,0.000000,27.404795
2,0.0,1.0,0.565792,0,0.000034,41.019178
3,0.0,0.0,0.684364,0,0.000000,37.500000
4,0.0,0.0,0.503142,0,0.000199,53.000000
...,...,...,...,...,...,...
134,1.0,0.0,0.742102,1,0.000000,0.000000
135,1.0,1.0,0.722918,1,0.000000,0.000000
136,1.0,0.0,0.652963,1,0.000000,39.498630
137,1.0,1.0,0.724255,1,0.000000,71.527397


In [26]:
X_new_std

,hpv_related,uicc8_III-IV,shape_Sphericity,oropharynx,LBP_102_PET,pack_years
0,0.0,0.0,0.704475,1,0.000000,-0.747766
1,0.0,0.0,0.101791,0,0.000000,0.159774
2,0.0,1.0,-1.132016,0,0.524263,0.610629
3,0.0,0.0,-0.017443,0,0.000000,0.494088
4,0.0,0.0,-1.720923,0,3.028668,1.007387
...,...,...,...,...,...,...
134,1.0,0.0,0.525285,1,0.000000,-0.747766
135,1.0,1.0,0.344965,1,0.000000,-0.747766
136,1.0,0.0,-0.312609,1,0.000000,0.560275
137,1.0,1.0,0.357529,1,0.000000,1.620942


In [27]:
MAASTRO_new

,hpv_related,uicc8_III-IV,shape_Sphericity,oropharynx,LBP_102_PET,pack_years
0,1,0,0.668072,1,0.000026,0
1,0,1,0.669961,1,0.000167,20
2,0,1,0.624081,1,0.000057,6
3,0,1,0.577624,0,0.000000,45
4,1,0,0.630933,1,0.000000,59
...,...,...,...,...,...,...
94,0,1,0.671754,0,0.000000,55
95,0,1,0.632189,0,0.000000,174
96,1,1,0.645548,1,0.000000,0
97,1,0,0.727488,1,0.000000,0


In [28]:
MAASTRO_new_std

,hpv_related,uicc8_III-IV,shape_Sphericity,oropharynx,LBP_102_PET,pack_years
0,1,0,-0.170593,1,0.398737,-0.747766
1,0,1,-0.152832,1,2.544568,-0.085444
2,0,1,-0.584101,1,0.869691,-0.549070
3,0,1,-1.020797,0,0.000000,0.742458
4,1,0,-0.519690,1,0.000000,1.206084
...,...,...,...,...,...,...
94,0,1,-0.135979,0,0.000000,1.073619
95,0,1,-0.507883,0,0.000000,5.014436
96,1,1,-0.382318,1,0.000000,-0.747766
97,1,0,0.387916,1,0.000000,-0.747766


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [29]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-19 14:32:13,839] A new study created in memory with name: no-name-13674658-0219-4cbd-aaf4-447381815ed8


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.8354978354978355
Fold 2 C-index: 0.7946428571428571


[I 2024-04-19 14:32:21,239] A new study created in memory with name: no-name-b20e5e83-0fa6-4158-bb7a-2d16a468160b


Fold 3 C-index: 0.7745098039215687
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 14:32:21,220] Trial 0 finished with value: 0.7737571035318616 and parameters: {}. Best is trial 0 with value: 0.7737571035318616.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7737571035318616], datetime_start=datetime.datetime(2024, 4, 19, 14, 32, 14, 133870), datetime_complete=datetime.datetime(2024, 4, 19, 14, 32, 21, 219469), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7737571035318616


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.1373367791207223
Fold 2 IBS: 0.17153895698332647
Fold 3 IBS: 0.18224864083271736
Fold 4 IBS: 0.18669684505649792
Fold 5 IBS: 0.22978024900389662
[I 2024-04-19 14:32:21,611] Trial 0 finished with value: 0.1815202941994321 and parameters: {}. Best is trial 0 with value: 0.1815202941994321.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.1815202941994321], datetime_start=datetime.datetime(2024, 4, 19, 14, 32, 21, 275452), datetime_complete=datetime.datetime(2024, 4, 19, 14, 32, 21, 610999), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.1815202941994321


In [30]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [31]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.774
train_ibs:  0.182


#### Test

In [32]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [33]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.587
IBS score: 0.278


In [34]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [35]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [36]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 14:32:22,372] A new study created in memory with name: no-name-a6b0fcf1-f098-4082-831e-73b8f6d50901


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-19 14:32:22,654] A new study created in memory with name: no-name-716049f5-6614-4379-a550-d6e5f7898c03


Fold 1 C-index: 0.6861471861471862
Fold 2 C-index: 0.7209821428571429
Fold 3 C-index: 0.6911764705882353
Fold 4 C-index: 0.6666666666666666
Fold 5 C-index: 0.6924882629107981
[I 2024-04-19 14:32:22,648] Trial 0 finished with value: 0.6914921458340058 and parameters: {}. Best is trial 0 with value: 0.6914921458340058.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6914921458340058], datetime_start=datetime.datetime(2024, 4, 19, 14, 32, 22, 428441), datetime_complete=datetime.datetime(2024, 4, 19, 14, 32, 22, 648269), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6914921458340058


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2139765170081932
Fold 2 IBS: 0.22157791027035073
Fold 3 IBS: 0.20453594896703353
Fold 4 IBS: 0.2247380375796925
Fold 5 IBS: 0.2181243109094921
[I 2024-04-19 14:32:22,983] Trial 0 finished with value: 0.21659054494695237 and parameters: {}. Best is trial 0 with value: 0.21659054494695237.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054494695237], datetime_start=datetime.datetime(2024, 4, 19, 14, 32, 22, 745361), datetime_complete=datetime.datetime(2024, 4, 19, 14, 32, 22, 983244), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054494695237


In [37]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [38]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.691
train_ibs:  0.217


#### Test

In [39]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [40]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.558


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [41]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [42]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 14:32:23,624] A new study created in memory with name: no-name-619d60c6-f71e-4f41-a103-65df373ddd27


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.7901785714285714
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.7974683544303798


[I 2024-04-19 14:32:24,116] A new study created in memory with name: no-name-1dbdf3da-708a-484d-8e4a-f1617fceb1eb


Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 14:32:24,104] Trial 0 finished with value: 0.7729788376800661 and parameters: {}. Best is trial 0 with value: 0.7729788376800661.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7729788376800661], datetime_start=datetime.datetime(2024, 4, 19, 14, 32, 23, 720353), datetime_complete=datetime.datetime(2024, 4, 19, 14, 32, 24, 104027), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7729788376800661


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.1379525564764891
Fold 2 IBS: 0.17259998000254334
Fold 3 IBS: 0.18072683268151377
Fold 4 IBS: 0.18569936734572245
Fold 5 IBS: 0.22816096227886445
[I 2024-04-19 14:32:24,768] Trial 0 finished with value: 0.18102793975702663 and parameters: {}. Best is trial 0 with value: 0.18102793975702663.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.18102793975702663], datetime_start=datetime.datetime(2024, 4, 19, 14, 32, 24, 189204), datetime_complete=datetime.datetime(2024, 4, 19, 14, 32, 24, 768195), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.18102793975702663


In [43]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [44]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.773
train_ibs:  0.181


#### Test 

In [45]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [46]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.59


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.273


In [47]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [48]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 14:32:25,251] A new study created in memory with name: no-name-9e6e320c-b9aa-4b05-9a1d-f08dcb0388cf


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 14:32:25,849] Trial 0 finished with value: 0.7720859805372091 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7720859805372091.
Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 14:32:26,334] Trial 1 finished with value: 0.771193123394352 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7720859805372091.
Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 14:32:26,901] Trial 2 finished with value: 0.7721735155512147 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with v

Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 14:32:39,364] Trial 24 finished with value: 0.7721735155512147 and parameters: {'l1_ratio': 0.17175907635020055}. Best is trial 14 with value: 0.7730663726940719.
Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 14:32:40,118] Trial 25 finished with value: 0.771193123394352 and parameters: {'l1_ratio': 0.3033674563887191}. Best is trial 14 with value: 0.7730663726940719.
Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 14:32:41,480] Trial 26 finished with value: 0.7720859805372091 and parameters: {'l1_ratio': 0.07348485814554673}. Best is trial 14 

Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 14:33:02,148] Trial 48 finished with value: 0.7720859805372091 and parameters: {'l1_ratio': 0.7768768486379835}. Best is trial 14 with value: 0.7730663726940719.
Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 14:33:03,054] Trial 49 finished with value: 0.7721735155512147 and parameters: {'l1_ratio': 0.1545469903233332}. Best is trial 14 with value: 0.7730663726940719.
Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 14:33:03,480] Trial 50 finished with value: 0.7720859805372091 and parameters: {'l1_ratio': 0.6064718647459368}. Best is trial 14 with value: 0.77306637269

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.13797076460579477
Fold 2 IBS: 0.17237767238545068
Fold 3 IBS: 0.1806649161740025
Fold 4 IBS: 0.18572867986550837
Fold 5 IBS: 0.22808900243690983
[I 2024-04-19 14:33:32,939] Trial 0 finished with value: 0.18096620709353323 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.18096620709353323.
Fold 1 IBS: 0.13797371195892438
Fold 2 IBS: 0.17327116971653533
Fold 3 IBS: 0.18062869781956573
Fold 4 IBS: 0.18575912095842703
Fold 5 IBS: 0.22793075681445532
[I 2024-04-19 14:33:33,502] Trial 1 finished with value: 0.18111269145358155 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.18096620709353323.
Fold 1 IBS: 0.1379785930621418
Fold 2 IBS: 0.17333767103026063
Fold 3 IBS: 0.18054443579449886
Fold 4 IBS: 0.18574442131862182
Fold 5 IBS: 0.22799850336404315
[I 2024-04-19 14:33:33,927] Trial 2 finished with value: 0.18112072491391323 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.18096620709

Fold 1 IBS: 0.1379339545809031
Fold 2 IBS: 0.17245799283377966
Fold 3 IBS: 0.1806645318212113
Fold 4 IBS: 0.18571532257538148
Fold 5 IBS: 0.22809951388565317
[I 2024-04-19 14:33:49,612] Trial 25 finished with value: 0.18097426313938575 and parameters: {'l1_ratio': 0.891573855041803}. Best is trial 11 with value: 0.1808929662541005.
Fold 1 IBS: 0.1379542845650902
Fold 2 IBS: 0.17275025042442463
Fold 3 IBS: 0.18068891302173123
Fold 4 IBS: 0.185735417795691
Fold 5 IBS: 0.22796451924283198
[I 2024-04-19 14:33:50,123] Trial 26 finished with value: 0.18101867700995383 and parameters: {'l1_ratio': 0.5479632818643458}. Best is trial 11 with value: 0.1808929662541005.
Fold 1 IBS: 0.21317323573569238
Fold 2 IBS: 0.22046998099861367
Fold 3 IBS: 0.18062771724693177
Fold 4 IBS: 0.22411935984071088
Fold 5 IBS: 0.22793413168944957
[I 2024-04-19 14:33:50,539] Trial 27 finished with value: 0.21326488510227964 and parameters: {'l1_ratio': 0.016560941611969082}. Best is trial 11 with value: 0.18089296625

Fold 2 IBS: 0.1722673980591959
Fold 3 IBS: 0.18064747294080516
Fold 4 IBS: 0.18569139236793886
Fold 5 IBS: 0.22806779841617478
[I 2024-04-19 14:34:04,590] Trial 50 finished with value: 0.1809230490312964 and parameters: {'l1_ratio': 0.7494606434304015}. Best is trial 11 with value: 0.1808929662541005.
Fold 1 IBS: 0.13796012934285062
Fold 2 IBS: 0.1722875465945281
Fold 3 IBS: 0.18070825626762008
Fold 4 IBS: 0.1857453921322719
Fold 5 IBS: 0.22813324336951613
[I 2024-04-19 14:34:05,280] Trial 51 finished with value: 0.18096691354135735 and parameters: {'l1_ratio': 0.7145748209714864}. Best is trial 11 with value: 0.1808929662541005.
Fold 1 IBS: 0.13796758000120174
Fold 2 IBS: 0.17236247178263364
Fold 3 IBS: 0.18065242499042497
Fold 4 IBS: 0.1857456486956535
Fold 5 IBS: 0.22806991991642278
[I 2024-04-19 14:34:06,273] Trial 52 finished with value: 0.18095960907726732 and parameters: {'l1_ratio': 0.8162426669043343}. Best is trial 11 with value: 0.1808929662541005.
Fold 1 IBS: 0.137926976523

Fold 2 IBS: 0.17228804826669653
Fold 3 IBS: 0.18068744438450812
Fold 4 IBS: 0.18570668216898698
Fold 5 IBS: 0.22811019959471515
[I 2024-04-19 14:34:24,788] Trial 75 finished with value: 0.18094491955371791 and parameters: {'l1_ratio': 0.7669378472415054}. Best is trial 71 with value: 0.18089114898863298.
Fold 1 IBS: 0.13795115831753071
Fold 2 IBS: 0.17222261005831627
Fold 3 IBS: 0.18060225198128796
Fold 4 IBS: 0.18567408752668435
Fold 5 IBS: 0.22801980479837397
[I 2024-04-19 14:34:25,553] Trial 76 finished with value: 0.18089398253643868 and parameters: {'l1_ratio': 0.7305334769066221}. Best is trial 71 with value: 0.18089114898863298.
Fold 1 IBS: 0.13793924934409754
Fold 2 IBS: 0.17249122456910568
Fold 3 IBS: 0.18069320770917602
Fold 4 IBS: 0.18568377175417458
Fold 5 IBS: 0.22811966221055005
[I 2024-04-19 14:34:26,330] Trial 77 finished with value: 0.18098542311742077 and parameters: {'l1_ratio': 0.6512273624253421}. Best is trial 71 with value: 0.18089114898863298.
Fold 1 IBS: 0.1379

In [49]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [50]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.773
train_ibs:  0.181


#### Test

In [51]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [52]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.1015556329868651)

test_cindex : 0.589


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.728038448531083)

test_ibs:  0.273


In [53]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [54]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-19 14:34:54,338] A new study created in memory with name: no-name-57ad5f98-2a45-44b7-a672-a3879b67e009


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.8268398268398268
Fold 2 C-index: 0.7232142857142857
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.7088607594936709
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 14:35:02,406] Trial 0 finished with value: 0.7439398371546546 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7439398371546546.
Fold 1 C-index: 0.8138528138528138
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.6995305164319249
[I 2024-04-19 14:35:08,717] Trial 1 finished with value: 0.7591482736873416 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 

Fold 5 C-index: 0.7417840375586855
[I 2024-04-19 14:36:30,990] Trial 15 finished with value: 0.8120274196000707 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 7, 'n_estimators': 480, 'oob_score': True, 'max_samples': 0.4470849120697906, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.08807762154599924, 'warm_start': True}. Best is trial 15 with value: 0.8120274196000707.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 14:36:37,388] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 7, 'min_samples_leaf': 1, 'max_depth': 7, 'n_estimators': 498, 'oob_score': True, 'max_samples': 0.36871324569404207, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.20791692251205332, 'warm_start': True}. Best is trial 15 with value: 0.8120274196000707.
Fold 1 C-index: 0.8181818181818182
Fold 2 C-index: 0.8125
Fold 3 C-index: 0.83333333333

Fold 3 C-index: 0.7965686274509803
Fold 4 C-index: 0.770042194092827
Fold 5 C-index: 0.7652582159624414
[I 2024-04-19 14:37:59,024] Trial 30 finished with value: 0.7825128767653189 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 6, 'min_samples_leaf': 6, 'max_depth': 1, 'n_estimators': 225, 'oob_score': True, 'max_samples': 0.5856012699634701, 'max_features': None, 'min_weight_fraction_leaf': 0.029048761893088054, 'warm_start': True}. Best is trial 19 with value: 0.8265809184741336.
Fold 1 C-index: 0.8268398268398268
Fold 2 C-index: 0.8303571428571429
Fold 3 C-index: 0.8284313725490197
Fold 4 C-index: 0.810126582278481
Fold 5 C-index: 0.755868544600939
[I 2024-04-19 14:38:05,474] Trial 31 finished with value: 0.810324693825082 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 4, 'min_samples_leaf': 11, 'max_depth': 4, 'n_estimators': 339, 'oob_score': True, 'max_samples': 0.5365082685042595, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.0654723504402752, 'wa

...

Fold 5 IBS: 0.2171431998129361
[I 2024-04-19 14:43:08,614] Trial 31 finished with value: 0.17792400393817337 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 17, 'min_samples_leaf': 6, 'max_depth': 4, 'n_estimators': 310, 'oob_score': False, 'max_samples': 0.6683617295863028, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.11208498314187895}. Best is trial 12 with value: 0.17673667335392534.
Fold 1 IBS: 0.14711946655783312
Fold 2 IBS: 0.19006811981061228
Fold 3 IBS: 0.158033991341854
Fold 4 IBS: 0.17821885585463562
Fold 5 IBS: 0.21747675580865514
[I 2024-04-19 14:43:14,025] Trial 32 finished with value: 0.17818343787471802 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 20, 'min_samples_leaf': 8, 'max_depth': 5, 'n_estimators': 310, 'oob_score': False, 'max_samples': 0.6782402761258088, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.10884764482583958}. Best is trial 12 with value: 0.17673667335392534.
Fold 1 IBS: 0.1474328186504624
Fold 2 IBS: 0.1925

Fold 1 IBS: 0.16189795049200065
Fold 2 IBS: 0.2002316521231612
Fold 3 IBS: 0.1632441421719029
Fold 4 IBS: 0.17861160833008888
Fold 5 IBS: 0.21900022601901006
[I 2024-04-19 14:44:01,299] Trial 47 finished with value: 0.18459711582723273 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 4, 'max_depth': 1, 'n_estimators': 130, 'oob_score': True, 'max_samples': 0.5217274207107006, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.0488148159561139}. Best is trial 38 with value: 0.17659645958662992.
Fold 1 IBS: 0.15250696953387413
Fold 2 IBS: 0.1814915271847653
Fold 3 IBS: 0.16047524829592633
Fold 4 IBS: 0.17246364113740187
Fold 5 IBS: 0.2120974321417849
[I 2024-04-19 14:44:02,810] Trial 48 finished with value: 0.1758069636587505 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 18, 'min_samples_leaf': 10, 'max_depth': 2, 'n_estimators': 70, 'oob_score': False, 'max_samples': 0.5834369548668846, 'max_features': 'sqrt', 'min_weight_fraction_leaf

Fold 1 IBS: 0.1528522544145544
Fold 2 IBS: 0.1883938076545454
Fold 3 IBS: 0.16059370160325126
Fold 4 IBS: 0.17447839059286202
Fold 5 IBS: 0.21675029793203757
[I 2024-04-19 14:44:26,224] Trial 63 finished with value: 0.17861369043945013 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 6, 'min_samples_leaf': 11, 'max_depth': 6, 'n_estimators': 121, 'oob_score': False, 'max_samples': 0.5082465499234735, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.0848024624668733}. Best is trial 48 with value: 0.1758069636587505.
Fold 1 IBS: 0.16166621880208873
Fold 2 IBS: 0.1731632023212488
Fold 3 IBS: 0.1700340266068216
Fold 4 IBS: 0.17478280891532624
Fold 5 IBS: 0.2135771701385966
[I 2024-04-19 14:44:26,883] Trial 64 finished with value: 0.17864468535681638 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 5, 'min_samples_leaf': 14, 'max_depth': 4, 'n_estimators': 16, 'oob_score': False, 'max_samples': 0.588577514842217, 'max_features': 'sqrt', 'min_weight_fraction_leaf':

Fold 1 IBS: 0.14805872849452492
Fold 2 IBS: 0.1844394959428906
Fold 3 IBS: 0.1592084284475042
Fold 4 IBS: 0.18049902936533177
Fold 5 IBS: 0.21718793826468474
[I 2024-04-19 14:44:42,698] Trial 79 finished with value: 0.17787872410298725 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 12, 'min_samples_leaf': 10, 'max_depth': 11, 'n_estimators': 50, 'oob_score': False, 'max_samples': 0.8221486447141937, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.10873595340811325}. Best is trial 48 with value: 0.1758069636587505.
Fold 1 IBS: 0.15361491788372222
Fold 2 IBS: 0.19642745280621465
Fold 3 IBS: 0.16095616456284817
Fold 4 IBS: 0.22665346294232952
Fold 5 IBS: 0.2385594621067867
[I 2024-04-19 14:44:44,357] Trial 80 finished with value: 0.19524229206038027 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 41, 'oob_score': False, 'max_samples': 0.7659640622029704, 'max_features': None, 'min_weight_fraction_leaf

Fold 5 IBS: 0.21759783800196422
[I 2024-04-19 14:45:08,700] Trial 94 finished with value: 0.1776435259600116 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 12, 'min_samples_leaf': 9, 'max_depth': 5, 'n_estimators': 119, 'oob_score': False, 'max_samples': 0.7242593013645032, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.14462066268912804}. Best is trial 81 with value: 0.17538981813433546.
Fold 1 IBS: 0.15162608620389867
Fold 2 IBS: 0.19121869268171396
Fold 3 IBS: 0.15836366628343707
Fold 4 IBS: 0.17826802865577213
Fold 5 IBS: 0.21920067113279704
[I 2024-04-19 14:45:11,366] Trial 95 finished with value: 0.17973542899152378 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 8, 'max_depth': 4, 'n_estimators': 137, 'oob_score': False, 'max_samples': 0.772556078583092, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.1631127999140187}. Best is trial 81 with value: 0.17538981813433546.
Fold 1 IBS: 0.16352525950573188
Fold 2 IBS: 0.1829

In [55]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [56]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.871
train_ibs:  0.175


#### Test

In [57]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [58]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=14, max_features='auto', max_leaf_nodes=15,
                     max_samples=0.9504102015210659, min_samples_leaf=2,
                     min_samples_split=9,
                     min_weight_fraction_leaf=0.005622697057305325,
                     n_estimators=144, random_state=123, warm_start=True)

test_cindex:  0.618


RandomSurvivalForest(max_depth=6, max_leaf_nodes=10,
                     max_samples=0.5785272262773831, min_samples_leaf=8,
                     min_samples_split=11,
                     min_weight_fraction_leaf=0.08332907721465231,
                     random_state=123)

test_ibs:  0.223


In [59]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [60]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [61]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 14:45:20,932] A new study created in memory with name: no-name-972bb78b-4747-4189-84ef-0d698e051982


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.8225108225108225
Fold 2 C-index: 0.7589285714285714
Fold 3 C-index: 0.8333333333333334
Fold 4 C-index: 0.8016877637130801
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 14:45:22,324] Trial 0 finished with value: 0.7766254315304948 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7766254315304948.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 14:45:25,996] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.8181818181818182
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.8161764705882353
Fold 4 C-index: 0.7932489451476793
Fold 5 C-index: 0.6572769953051644
[I 2024-04-19 14:46:10,188] Trial 16 finished with value: 0.7660839887017223 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6535355702555379, 'min_weight_fraction_leaf': 0.1906011147608997}. Best is trial 12 with value: 0.786055160528368.
Fold 1 C-index: 0.8333333333333334
Fold 2 C-index: 0.7834821428571429
Fold 3 C-index: 0.8063725490196079
Fold 4 C-index: 0.8080168776371308
Fold 5 C-index: 0.6314553990610329
[I 2024-04-19 14:46:11,970] Trial 17 finished with value: 0.7725320603816497 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 10, 'n_estimators': 384, 'oob_score': False, 'warm_start': True, 'max_featur

Fold 1 C-index: 0.8268398268398268
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.8284313725490197
Fold 4 C-index: 0.8227848101265823
Fold 5 C-index: 0.676056338028169
[I 2024-04-19 14:46:38,623] Trial 31 finished with value: 0.7852867552230053 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 11, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 204, 'oob_score': False, 'warm_start': True, 'max_features': 1, 'max_samples': 0.8438862458923003, 'min_weight_fraction_leaf': 0.03902180415717787}. Best is trial 25 with value: 0.7919233215012451.
Fold 1 C-index: 0.8181818181818182
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.8284313725490197
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.676056338028169
[I 2024-04-19 14:46:39,990] Trial 32 finished with value: 0.7827112716348635 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 12, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 283, 'oob_score': False, 'warm_start': True, 'max_features':

...

Fold 1 IBS: 0.1585529260071
Fold 2 IBS: 0.218153010449042
Fold 3 IBS: 0.17021896151952115
Fold 4 IBS: 0.16343124711768475
Fold 5 IBS: 0.2311751417053347
[I 2024-04-19 14:50:03,040] Trial 30 finished with value: 0.18830625735973652 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 6, 'max_depth': 3, 'n_estimators': 116, 'oob_score': False, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.8258632202024414, 'min_weight_fraction_leaf': 0.20241101766143602}. Best is trial 7 with value: 0.1857272253521116.
Fold 1 IBS: 0.15420880459120728
Fold 2 IBS: 0.22137976081415967
Fold 3 IBS: 0.16774275391760834
Fold 4 IBS: 0.15688252726853427
Fold 5 IBS: 0.23380379197752338
[I 2024-04-19 14:50:05,523] Trial 31 finished with value: 0.18680352771380657 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 17, 'n_estimators': 108, 'oob_score': False, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.61

Fold 1 IBS: 0.1350712935921733
Fold 2 IBS: 0.21575795396259614
Fold 3 IBS: 0.1719895889282686
Fold 4 IBS: 0.1639662174932425
Fold 5 IBS: 0.23179177468347978
[I 2024-04-19 14:52:01,489] Trial 45 finished with value: 0.18371536573195205 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 6, 'min_samples_leaf': 2, 'max_depth': 8, 'n_estimators': 446, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.7218458884001862, 'min_weight_fraction_leaf': 0.002908146688295732}. Best is trial 42 with value: 0.18220424087625745.
Fold 1 IBS: 0.14405258172603636
Fold 2 IBS: 0.23510729165077407
Fold 3 IBS: 0.1652427330079092
Fold 4 IBS: 0.15610840831496184
Fold 5 IBS: 0.23462655467653304
[I 2024-04-19 14:52:10,135] Trial 46 finished with value: 0.1870275138752429 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 4, 'min_samples_leaf': 2, 'max_depth': 12, 'n_estimators': 302, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.77768969

Fold 1 IBS: 0.2139754605574802
Fold 2 IBS: 0.22128037493850963
Fold 3 IBS: 0.2049770044390806
Fold 4 IBS: 0.22468785016066256
Fold 5 IBS: 0.21820838836670314
[I 2024-04-19 14:54:20,961] Trial 60 finished with value: 0.21662581569248723 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 9, 'n_estimators': 373, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.36612707136592737, 'min_weight_fraction_leaf': 0.2583710069782479}. Best is trial 42 with value: 0.18220424087625745.
Fold 1 IBS: 0.13013980993149055
Fold 2 IBS: 0.21254887098310069
Fold 3 IBS: 0.17298441817166582
Fold 4 IBS: 0.16679488873081466
Fold 5 IBS: 0.2331102795997352
[I 2024-04-19 14:54:31,158] Trial 61 finished with value: 0.18311565348336137 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 7, 'min_samples_leaf': 2, 'max_depth': 8, 'n_estimators': 439, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.77533595

Fold 1 IBS: 0.1331805487459175
Fold 2 IBS: 0.21624338660783993
Fold 3 IBS: 0.1727837835092345
Fold 4 IBS: 0.16512709360909275
Fold 5 IBS: 0.23118212185240014
[I 2024-04-19 14:56:41,836] Trial 75 finished with value: 0.18370338686489696 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 7, 'min_samples_leaf': 2, 'max_depth': 11, 'n_estimators': 388, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.8812725582045702, 'min_weight_fraction_leaf': 0.014257699585348107}. Best is trial 42 with value: 0.18220424087625745.
Fold 1 IBS: 0.14741275991375746
Fold 2 IBS: 0.22339111860096061
Fold 3 IBS: 0.16719852475388572
Fold 4 IBS: 0.1582361988967138
Fold 5 IBS: 0.23150079709187427
[I 2024-04-19 14:56:51,924] Trial 76 finished with value: 0.18554787985143836 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 8, 'min_samples_leaf': 1, 'max_depth': 10, 'n_estimators': 446, 'oob_score': True, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.96

Fold 1 IBS: 0.12988717417010312
Fold 2 IBS: 0.21111739221808115
Fold 3 IBS: 0.17281260544267968
Fold 4 IBS: 0.16744968489433068
Fold 5 IBS: 0.23530465742353276
[I 2024-04-19 14:59:02,170] Trial 90 finished with value: 0.18331430282974548 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 10, 'n_estimators': 317, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.7774148609541289, 'min_weight_fraction_leaf': 0.011005982719496389}. Best is trial 42 with value: 0.18220424087625745.
Fold 1 IBS: 0.12951620582034926
Fold 2 IBS: 0.210867217186133
Fold 3 IBS: 0.17327356003962616
Fold 4 IBS: 0.16765928672732763
Fold 5 IBS: 0.2351767398201044
[I 2024-04-19 14:59:09,838] Trial 91 finished with value: 0.1832986019187081 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 10, 'n_estimators': 322, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.77818

In [62]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [63]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.816
train_ibs:  0.181


#### Test

In [64]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [65]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=11, max_leaf_nodes=13,
                   max_samples=0.9554816381741302, min_samples_leaf=1,
                   min_samples_split=17,
                   min_weight_fraction_leaf=0.026333169142875475,
                   n_estimators=68, random_state=123, warm_start=True)

C-index score: 0.656


ExtraSurvivalTrees(max_depth=13, max_features=1, max_leaf_nodes=11,
                   max_samples=0.7098772062937373, min_samples_leaf=1,
                   min_samples_split=14,
                   min_weight_fraction_leaf=0.012799049735992758,
                   n_estimators=288, oob_score=True, random_state=123,
                   warm_start=True)

IBS: 0.214


In [66]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [67]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-19 15:00:09,672] A new study created in memory with name: no-name-be8abf21-629a-4b4c-9349-1d82acb93c86


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 15:00:46,667] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 15:01:06,549] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 15:12:21,209] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7471201964999683.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 15:13:48,472] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 15:28:00,280] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 22 with value: 0.7520782207516631.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5558035714285714
Fold 3 C-index: 0.6666666666666666
Fold 4 C-index: 0.5654008438818565
Fold 5 C-index: 0.636150234741784
[I 2024-04-19 15:29:38,781] Trial 26 finished with value: 0.5848042633437757 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropo

...

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-19 17:19:57,511] Trial 55 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.918589848048704, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.1366452847323028, 'n_estimators': 388, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.18967222528443176, 'max_features': 'auto', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 15, 'max_leaf_nodes': 15, 'min_samples_leaf': 19, 'max_depth': 18}. Best is trial 53 with value: 0.21558143837167032.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609582
[I 2024-04-19 17:22:18,811] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.7513175598857118, 'learning_rate': 0.0985092209

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-19 18:59:51,238] Trial 66 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9998769156045215, 'learning_rate': 0.05359198006915804, 'dropout_rate': 0.650983074619535, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.713342732410399, 'min_weight_fraction_leaf': 0.037327349410482574, 'max_features': 0.1, 'min_impurity_decrease': 4.636270239719423e-07, 'validation_fraction': 0.7909756969364712, 'min_samples_split': 9, 'max_leaf_nodes': 8, 'min_samples_leaf': 18, 'max_depth': 19}. Best is trial 64 with value: 0.21551873707634944.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.2181243152560957
[I 2024-04-19 19:01:31,852] Trial 67 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8945047973351204, 'learning_rate': 0.03028968218872250

Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-19 19:23:57,458] Trial 77 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.6527746876723795, 'learning_rate': 0.009617814274368723, 'dropout_rate': 0.23464360454432553, 'n_estimators': 449, 'criterion': 'squared_error', 'ccp_alpha': 0.5872217111247551, 'min_weight_fraction_leaf': 0.12767889602965304, 'max_features': 1, 'min_impurity_decrease': 1.4509408420175117e-05, 'validation_fraction': 0.9291496186906082, 'min_samples_split': 8, 'max_leaf_nodes': 6, 'min_samples_leaf': 20, 'max_depth': 5}. Best is trial 72 with value: 0.21520875346396268.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609557
[I 2024-04-19 19:27:03,494] Trial 78 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9412481314247185, 'learning_rate': 0.01401320577538

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-19 19:54:35,143] Trial 88 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.7762725560789899, 'learning_rate': 0.011111179381597284, 'dropout_rate': 0.3196243460718578, 'n_estimators': 407, 'criterion': 'squared_error', 'ccp_alpha': 1.3457840140168202, 'min_weight_fraction_leaf': 0.026102976372990007, 'max_features': 0.1, 'min_impurity_decrease': 4.1842504540759167e-07, 'validation_fraction': 0.8468844412711507, 'min_samples_split': 5, 'max_leaf_nodes': 20, 'min_samples_leaf': 7, 'max_depth': 8}. Best is trial 72 with value: 0.21520875346396268.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-19 19:56:47,409] Trial 89 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.6969649633096852, 'learning_rate': 0.01535870401

Fold 3 IBS: 0.2023614521860876
Fold 4 IBS: 0.2231898890507258
Fold 5 IBS: 0.2170482819994517
[I 2024-04-19 20:27:39,066] Trial 99 finished with value: 0.2148526864616871 and parameters: {'subsample': 0.8333920238412017, 'learning_rate': 0.02538174390008161, 'dropout_rate': 0.4577965489047534, 'n_estimators': 399, 'criterion': 'squared_error', 'ccp_alpha': 0.004367047248614303, 'min_weight_fraction_leaf': 0.18072249935794454, 'max_features': 'auto', 'min_impurity_decrease': 4.472271308488144e-07, 'validation_fraction': 0.9580200141092102, 'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 14, 'max_depth': 5}. Best is trial 99 with value: 0.2148526864616871.


* Best trial for IBS: 
 FrozenTrial(number=99, state=TrialState.COMPLETE, values=[0.2148526864616871], datetime_start=datetime.datetime(2024, 4, 19, 20, 25, 57, 706241), datetime_complete=datetime.datetime(2024, 4, 19, 20, 27, 39, 65185), params={'subsample': 0.8333920238412017, 'learning_rate': 0.02538174390008161, 

In [68]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [69]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.791
train_ibs:  0.215


#### Test

In [70]:
  # y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [71]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.030433464476024484,
                                 criterion='squared_error',
                                 dropout_rate=0.6934300990357392,
                                 learning_rate=0.05396213117321635, max_depth=7,
                                 max_features=1, max_leaf_nodes=10,
                                 min_impurity_decrease=3.7676942671090066e-07,
                                 min_samples_leaf=11, min_samples_split=20,
                                 min_weight_fraction_leaf=0.38729753243593296,
                                 n_estimators=424, random_state=123,
                                 subsample=0.38604321258984187,
                                 validation_fraction=0.9213195792266449)

C-index score: 0.63


GradientBoostingSurvivalAnalysis(ccp_alpha=0.004367047248614303,
                                 criterion='squared_error',
                                 dropout_rate=0.4577965489047534,
                                 learning_rate=0.02538174390008161, max_depth=5,
                                 max_features='auto', max_leaf_nodes=17,
                                 min_impurity_decrease=4.472271308488144e-07,
                                 min_samples_leaf=14, min_samples_split=3,
                                 min_weight_fraction_leaf=0.18072249935794454,
                                 n_estimators=399, random_state=123,
                                 subsample=0.8333920238412017,
                                 validation_fraction=0.9580200141092102)

IBS: 0.22


In [72]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [73]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [74]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 20:28:14,176] A new study created in memory with name: no-name-015d58cd-203a-4156-9e43-726d6691d6eb


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7352941176470589
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-19 20:28:16,362] Trial 0 finished with value: 0.7136894859079793 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.7136894859079793.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7352941176470589
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-19 20:28:36,752] Trial 1 finished with value: 0.7136894859079793 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.7136894859079793.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7352941176470589
Fold 4 C-index: 0.6624472573839663
Fold 5 C-ind

Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.7401960784313726
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7230046948356808
[I 2024-04-19 20:31:32,814] Trial 19 finished with value: 0.7118990866496844 and parameters: {'subsample': 0.37351945568220557, 'dropout_rate': 0.7683864204868823, 'n_estimators': 431, 'learning_rate': 0.08253305039935499}. Best is trial 13 with value: 0.7216408865771629.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7589285714285714
Fold 3 C-index: 0.7450980392156863
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7230046948356808
[I 2024-04-19 20:31:43,779] Trial 20 finished with value: 0.7164238511009196 and parameters: {'subsample': 0.179916923106687, 'dropout_rate': 0.5923165972699604, 'n_estimators': 337, 'learning_rate': 0.03973446561107324}. Best is trial 13 with value: 0.7216408865771629.
Fold 1 C-index: 0.696969696969697
Fold 2 C-index: 0.7589285714285714
Fold 3 C-index: 0.7598039215686274
Fo

Fold 1 C-index: 0.696969696969697
Fold 2 C-index: 0.7589285714285714
Fold 3 C-index: 0.7450980392156863
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7089201877934272
[I 2024-04-19 20:35:24,601] Trial 38 finished with value: 0.7144727505582698 and parameters: {'subsample': 0.2426785762118624, 'dropout_rate': 0.13272164755980653, 'n_estimators': 261, 'learning_rate': 0.07341163028968031}. Best is trial 21 with value: 0.7217284215911685.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7589285714285714
Fold 3 C-index: 0.7401960784313726
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.704225352112676
[I 2024-04-19 20:35:27,994] Trial 39 finished with value: 0.7116875903994557 and parameters: {'subsample': 0.19457273105148687, 'dropout_rate': 0.545457595342022, 'n_estimators': 145, 'learning_rate': 0.08805878681634485}. Best is trial 21 with value: 0.7217284215911685.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.7401960784313726
Fo

...

Fold 2 IBS: 0.18403148263985214
Fold 3 IBS: 0.1794640168289826
Fold 4 IBS: 0.20530510535013138
Fold 5 IBS: 0.19099563016620663
[I 2024-04-19 20:51:48,167] Trial 19 finished with value: 0.1893596887475797 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 19 with value: 0.1893596887475797.
Fold 1 IBS: 0.19149264401294455
Fold 2 IBS: 0.18706319876174624
Fold 3 IBS: 0.18151616590927064
Fold 4 IBS: 0.20627633661512726
Fold 5 IBS: 0.19535824476576305
[I 2024-04-19 20:51:49,274] Trial 20 finished with value: 0.19234131801297033 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.1893596887475797.
Fold 1 IBS: 0.19131235337562966
Fold 2 IBS: 0.19128086039557748
Fold 3 IBS: 0.1841970439756356
Fold 4 IBS: 0.2060462029855898
Fold 5 IBS: 0.19662556381580787
[I 2024-04-

Fold 3 IBS: 0.19014918624192317
Fold 4 IBS: 0.24978463432611944
Fold 5 IBS: 0.19584476340890472
[I 2024-04-19 20:52:35,959] Trial 38 finished with value: 0.20777727559225453 and parameters: {'subsample': 0.24276783442088262, 'dropout_rate': 0.13272164755980653, 'n_estimators': 87, 'learning_rate': 0.06952328337469094}. Best is trial 19 with value: 0.1893596887475797.
Fold 1 IBS: 0.19338706777728606
Fold 2 IBS: 0.18922224442798738
Fold 3 IBS: 0.18465392240622586
Fold 4 IBS: 0.2095057018732719
Fold 5 IBS: 0.1917921429574166
[I 2024-04-19 20:52:36,838] Trial 39 finished with value: 0.19371221588843754 and parameters: {'subsample': 0.3645779440068705, 'dropout_rate': 0.9451944095204212, 'n_estimators': 32, 'learning_rate': 0.055414173921640594}. Best is trial 19 with value: 0.1893596887475797.
Fold 1 IBS: 0.1938766062087041
Fold 2 IBS: 0.19294131275618495
Fold 3 IBS: 0.18743576949243462
Fold 4 IBS: 0.20765416654906
Fold 5 IBS: 0.1966197339957858
[I 2024-04-19 20:52:37,918] Trial 40 finishe

Fold 4 IBS: 0.22689079236666937
Fold 5 IBS: 0.18664061340044533
[I 2024-04-19 20:53:08,794] Trial 57 finished with value: 0.19984574725258203 and parameters: {'subsample': 0.5941008044918863, 'dropout_rate': 0.8145211095831743, 'n_estimators': 76, 'learning_rate': 0.04361786189303573}. Best is trial 19 with value: 0.1893596887475797.
Fold 1 IBS: 0.2682466343232404
Fold 2 IBS: 0.2787824438907526
Fold 3 IBS: 0.23921027812662599
Fold 4 IBS: 0.2777408841763473
Fold 5 IBS: 0.24597873082417693
[I 2024-04-19 20:53:22,905] Trial 58 finished with value: 0.26199179426822866 and parameters: {'subsample': 0.26673471749953837, 'dropout_rate': 0.971121079564111, 'n_estimators': 498, 'learning_rate': 0.01939068551573221}. Best is trial 19 with value: 0.1893596887475797.
Fold 1 IBS: 0.19376635289913924
Fold 2 IBS: 0.189719893658391
Fold 3 IBS: 0.18505133023430217
Fold 4 IBS: 0.20843304746625627
Fold 5 IBS: 0.1926255656956176
[I 2024-04-19 20:53:23,671] Trial 59 finished with value: 0.19391923799074126

Fold 5 IBS: 0.18850998676923908
[I 2024-04-19 20:54:02,874] Trial 76 finished with value: 0.19826989635275663 and parameters: {'subsample': 0.12630568358802877, 'dropout_rate': 0.8840084345145566, 'n_estimators': 124, 'learning_rate': 0.03271685219727702}. Best is trial 19 with value: 0.1893596887475797.
Fold 1 IBS: 0.18940730445015777
Fold 2 IBS: 0.18561723422967616
Fold 3 IBS: 0.1792229738858936
Fold 4 IBS: 0.2055836548297881
Fold 5 IBS: 0.1921637886082129
[I 2024-04-19 20:54:04,220] Trial 77 finished with value: 0.19039899120074572 and parameters: {'subsample': 0.10014660118586548, 'dropout_rate': 0.8443946246731343, 'n_estimators': 72, 'learning_rate': 0.02825007388476382}. Best is trial 19 with value: 0.1893596887475797.
Fold 1 IBS: 0.18932330340882508
Fold 2 IBS: 0.18926985095602739
Fold 3 IBS: 0.18192504040822394
Fold 4 IBS: 0.20484414608336163
Fold 5 IBS: 0.19515133549350383
[I 2024-04-19 20:54:05,714] Trial 78 finished with value: 0.1921027352699884 and parameters: {'subsample

Fold 5 IBS: 0.20014255507293266
[I 2024-04-19 20:54:41,246] Trial 95 finished with value: 0.2195129963843307 and parameters: {'subsample': 0.22312870945580054, 'dropout_rate': 0.18446795693693951, 'n_estimators': 157, 'learning_rate': 0.039295794278161086}. Best is trial 19 with value: 0.1893596887475797.
Fold 1 IBS: 0.19992479383400288
Fold 2 IBS: 0.20517405006053488
Fold 3 IBS: 0.19391089824160457
Fold 4 IBS: 0.21217526772368775
Fold 5 IBS: 0.20713449631139202
[I 2024-04-19 20:54:41,961] Trial 96 finished with value: 0.20366390123424444 and parameters: {'subsample': 0.1912179760870773, 'dropout_rate': 0.22408550586961065, 'n_estimators': 15, 'learning_rate': 0.04302122604055191}. Best is trial 19 with value: 0.1893596887475797.
Fold 1 IBS: 0.19731564490295572
Fold 2 IBS: 0.18212229764194593
Fold 3 IBS: 0.17940905442950114
Fold 4 IBS: 0.22963273523465397
Fold 5 IBS: 0.18885746554699534
[I 2024-04-19 20:54:44,111] Trial 97 finished with value: 0.19546743955121043 and parameters: {'subs

In [75]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [76]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.724
train_ibs:  0.189


#### Test

In [77]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [78]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.6140864163365967,
                                              learning_rate=0.09358014693300638,
                                              n_estimators=441,
                                              random_state=123,
                                              subsample=0.10718290653750012)

C-index score: 0.556


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.9930703343219778,
                                              learning_rate=0.04096883532946153,
                                              n_estimators=51, random_state=123,
                                              subsample=0.10013306718236009)

IBS: 0.229


In [79]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [80]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.871,1.0
ExtraSurvivalTrees,0.816,2.0
GradientBoosting,0.791,3.0
CoxPH,0.774,4.0
CoxLasso,0.773,5.5
CoxElastic,0.773,5.5
ComponentwiseGradientBoosting,0.724,7.0
CoxRidge,0.691,8.0


In [81]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.175,1.0
CoxLasso,0.181,3.0
CoxElastic,0.181,3.0
ExtraSurvivalTrees,0.181,3.0
CoxPH,0.182,5.0
ComponentwiseGradientBoosting,0.189,6.0
GradientBoosting,0.215,7.0
CoxRidge,0.217,8.0


In [82]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ExtraSurvivalTrees,0.656,1.0
GradientBoosting,0.630,2.0
Randomsurvivalforest,0.618,3.0
CoxLasso,0.590,4.0
CoxElastic,0.589,5.0
CoxPH,0.587,6.0
CoxRidge,0.558,7.0
ComponentwiseGradientBoosting,0.556,8.0


In [83]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
ExtraSurvivalTrees,0.214,1.0
GradientBoosting,0.220,2.0
CoxRidge,0.221,3.0
Randomsurvivalforest,0.223,4.0
ComponentwiseGradientBoosting,0.229,5.0
CoxLasso,0.273,6.5
CoxElastic,0.273,6.5
CoxPH,0.278,8.0


In [84]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/os/robust/rent/' 

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_os_robust_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [85]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-19
